# Hospital Database



In [20]:
# Hospital Generating Database

import sqlite3
import random
import datetime as dt

# 1. DB connection
# Change this path if your folder is different
db_path = r"C:\M\sqlite project\Hospital_Data_base.db"

connection = sqlite3.connect(db_path)
cursor = connection.cursor()

# Foreign keys ON
cursor.execute("PRAGMA foreign_keys = ON;")

# Fix random seed for reproducibility
random.seed(42)

print("Connected to:", db_path)


# 2. Create schema (all tables)

schema_sql = """
PRAGMA foreign_keys = ON;

-- 1. Hospital care units
CREATE TABLE IF NOT EXISTS care_units (
    unit_id      INTEGER PRIMARY KEY,
    unit_title   TEXT NOT NULL,
    unit_floor   INTEGER NOT NULL CHECK(unit_floor >= 0),
    phone_ext    TEXT
);

-- 2. Clinician registry
CREATE TABLE IF NOT EXISTS clinician_registry (
    clinician_id       INTEGER PRIMARY KEY,
    clinician_fullname TEXT NOT NULL,
    specialty_domain   TEXT NOT NULL,
    unit_ref_id        INTEGER NOT NULL,
    grade_band         TEXT,
    is_active          INTEGER NOT NULL CHECK(is_active IN (0,1)),
    FOREIGN KEY (unit_ref_id) REFERENCES care_units(unit_id)
);

-- 3. Patient profiles
CREATE TABLE IF NOT EXISTS client_profiles (
    client_id        INTEGER PRIMARY KEY,
    client_fullname  TEXT NOT NULL,
    biological_sex   TEXT NOT NULL CHECK (biological_sex IN ('Male','Female','Other')),
    age_in_years     INTEGER NOT NULL CHECK (age_in_years >= 0),
    height_cm        REAL CHECK (height_cm > 0),
    weight_kg        REAL CHECK (weight_kg > 0),
    blood_grouping   TEXT NOT NULL,
    postcode         TEXT NOT NULL,
    emergency_contact TEXT
);

-- 4. Clinical encounters
CREATE TABLE IF NOT EXISTS clinical_encounters (
    encounter_id     INTEGER PRIMARY KEY,
    client_ref_id    INTEGER NOT NULL,
    clinician_ref_id INTEGER NOT NULL,
    unit_ref_id      INTEGER NOT NULL,
    encounter_date   TEXT NOT NULL,
    encounter_time   TEXT NOT NULL,
    triage_level     INTEGER NOT NULL CHECK (triage_level BETWEEN 1 AND 5),
    pain_score       INTEGER CHECK (pain_score BETWEEN 0 AND 10),
    temperature_c    REAL,
    bp_systolic      INTEGER,
    bp_diastolic     INTEGER,
    FOREIGN KEY (client_ref_id)    REFERENCES client_profiles(client_id),
    FOREIGN KEY (clinician_ref_id) REFERENCES clinician_registry(clinician_id),
    FOREIGN KEY (unit_ref_id)      REFERENCES care_units(unit_id)
);

-- 5. Diagnostic events
CREATE TABLE IF NOT EXISTS diagnostic_events (
    diagnostic_id    INTEGER PRIMARY KEY,
    encounter_ref_id INTEGER NOT NULL,
    diagnostic_code  TEXT NOT NULL,
    severity_band    INTEGER NOT NULL CHECK (severity_band BETWEEN 1 AND 3),
    clinical_notes   TEXT,
    FOREIGN KEY (encounter_ref_id) REFERENCES clinical_encounters(encounter_id)
);

-- 6. Procedures
CREATE TABLE IF NOT EXISTS procedure_catalogue (
    encounter_ref_id  INTEGER NOT NULL,
    procedure_code    TEXT NOT NULL,
    procedure_label   TEXT NOT NULL,
    cost_in_currency  REAL NOT NULL CHECK (cost_in_currency >= 0),
    PRIMARY KEY (encounter_ref_id, procedure_code),
    FOREIGN KEY (encounter_ref_id) REFERENCES clinical_encounters(encounter_id)
);

-- 7. Medications
CREATE TABLE IF NOT EXISTS medication_orders (
    encounter_ref_id  INTEGER NOT NULL,
    drug_label        TEXT NOT NULL,
    dose_in_mg        REAL NOT NULL CHECK (dose_in_mg > 0),
    duration_days     INTEGER NOT NULL CHECK (duration_days > 0),
    PRIMARY KEY (encounter_ref_id, drug_label),
    FOREIGN KEY (encounter_ref_id) REFERENCES clinical_encounters(encounter_id)
);

-- 8. Billing
CREATE TABLE IF NOT EXISTS financial_statements (
    bill_id          INTEGER PRIMARY KEY,
    encounter_ref_id INTEGER NOT NULL,
    amount_payable   REAL NOT NULL CHECK (amount_payable >= 0),
    payment_flag     TEXT NOT NULL CHECK (payment_flag IN ('Paid','Pending')),
    settlement_date  TEXT,
    FOREIGN KEY (encounter_ref_id) REFERENCES clinical_encounters(encounter_id)
);
"""

cursor.executescript(schema_sql)
connection.commit()
print("Schema created.")


# 3. Insert fixed reference data (units + clinicians)

# Clear old data if re-running
cursor.execute("DELETE FROM clinician_registry;")
cursor.execute("DELETE FROM care_units;")
connection.commit()

# Insert care units
cursor.executemany("""
    INSERT INTO care_units (unit_id, unit_title, unit_floor, phone_ext)
    VALUES (?, ?, ?, ?);
""", [
    (1, 'Emergency',        0, '1001'),
    (2, 'Cardiology',       2, '1201'),
    (3, 'Neurology',        3, '1301'),
    (4, 'General Medicine', 1, '1105'),
    (5, 'Orthopaedics',     2, '1250')
])

# Insert clinicians
cursor.executemany("""
    INSERT INTO clinician_registry (
        clinician_id, clinician_fullname, specialty_domain,
        unit_ref_id, grade_band, is_active
    ) VALUES (?, ?, ?, ?, ?, ?);
""", [
    (1, 'Dr. Anil Kumar',    'Emergency Medicine', 1, 'Consultant', 1),
    (2, 'Dr. Sara Johnson',  'Cardiology',         2, 'Consultant', 1),
    (3, 'Dr. Naveen Reddy',  'Neurology',          3, 'Registrar',  1),
    (4, 'Dr. Priya Patel',   'General Medicine',   4, 'Consultant', 1),
    (5, 'Dr. Omar Ali',      'General Medicine',   4, 'Registrar',  1),
    (6, 'Dr. Emily Brown',   'Orthopaedics',       5, 'Consultant', 1),
    (7, 'Dr. Vikram Singh',  'Emergency Medicine', 1, 'Registrar',  1),
])

connection.commit()
print("Reference data inserted.")

# -------- 4. Helper lists + small functions --------

names = [
    "virat", "ravi", "sangeetha", "mahesh", "babu",
    "naveen", "ntr", "allu arjun", "aparna", "alluri",
    "kohli", "ramesh", "suresh", "navitha", "sreedhar",
    "chalapathi reddy", "ramu", "rachana", "sathish",
    "narayana", "ammu", "suchitra", "vinny",
    "vinay", "vishal", "vishwak"
]

postcode_prefixes = ["EC1", "EC2", "N1", "SE1", "SW1", "E14", "W1", "NW1"]

def random_postcode():
    prefix = random.choice(postcode_prefixes)
    number = random.randint(1, 9)
    letters = random.choice(["AA", "AB", "BA", "BB", "XZ", "ZX"])
    return f"{prefix} {number}{letters}"

clinician_to_unit = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 1
}

diagnosis_codes = [
    ("R50", "Acute fever"),
    ("I10", "Essential hypertension"),
    ("J45", "Bronchial asthma"),
    ("E11", "Type 2 diabetes"),
    ("J18", "Community-acquired pneumonia"),
    ("K52", "Gastroenteritis"),
    ("N39", "Urinary tract infection"),
    ("G40", "Epileptic seizure"),
    ("S06", "Head injury"),
    ("M54", "Back pain"),
    ("R07", "Chest pain"),
    ("F41", "Anxiety state")
]

procedure_list = [
    ("ECG",      "Electrocardiogram",                 60.0),
    ("CXR",      "Chest X-Ray",                       80.0),
    ("CTHEAD",   "CT Head",                          250.0),
    ("MRIHEAD",  "MRI Brain",                        450.0),
    ("BLOOD01",  "Full blood count",                  40.0),
    ("BLOOD02",  "Biochemistry panel",                55.0),
    ("USABD",    "Abdominal ultrasound",             180.0),
    ("DRES01",   "Wound dressing",                    35.0),
    ("PLASTER",  "Plaster cast application",         120.0),
    ("PHYSIO",   "Physiotherapy session",             75.0),
    ("ECHO",     "Echocardiogram",                   220.0),
    ("STRESS",   "Cardiac stress test",              260.0),
    ("NEURO",    "Nerve conduction study",           200.0)
]

drug_list = [
    "Paracetamol 500 mg tablet",
    "Ibuprofen 400 mg tablet",
    "Omeprazole 20 mg capsule",
    "Metformin 500 mg tablet",
    "Insulin injection",
    "Salbutamol inhaler",
    "Amoxicillin 500 mg capsule",
    "Sertraline 50 mg tablet",
    "Losartan 50 mg tablet",
    "Amlodipine 5 mg tablet",
    "Morphine injection",
    "Ondansetron injection",
    "Dexamethasone tablet"
]

def random_date(start_year=2021, end_year=2025):
    start = dt.date(start_year, 1, 1)
    end = dt.date(end_year, 11, 1)
    delta_days = (end - start).days
    random_day = start + dt.timedelta(days=random.randint(0, delta_days))
    return random_day.isoformat()

def random_time():
    h = random.randint(0, 23)
    m = random.randint(0, 59)
    s = random.randint(0, 59)
    return f"{h:02d}:{m:02d}:{s:02d}"

# -------- 5. Generate patients --------

def generate_clients(cursor, start_id=1, num_new=310):
    for client_id in range(start_id, start_id + num_new):
        full_name = random.choice(names).title() + " " + random.choice(names).title()
        sex = random.choice(["Male", "Female"])
        age = random.randint(0, 90)
        height = random.uniform(150, 190)
        weight = random.uniform(45, 120)
        blood_group = random.choice(["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"])
        postcode = random_postcode()

        if random.random() < 0.20:
            emergency = None
        else:
            emergency = random.choice(names).title() + " " + random.choice(names).title()

        cursor.execute("""
            INSERT INTO client_profiles (
                client_id, client_fullname, biological_sex,
                age_in_years, height_cm, weight_kg,
                blood_grouping, postcode, emergency_contact
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
        """, (
            client_id, full_name, sex,
            age, round(height, 1), round(weight, 1),
            blood_group, postcode, emergency
        ))

    print(f"Inserted {num_new} client records.")

cursor.execute("DELETE FROM client_profiles;")
connection.commit()
generate_clients(cursor, start_id=1, num_new=310)
connection.commit()

# 6. Generate encounters 

def generate_encounters(cursor, start_encounter_id=1, num_new=1015, max_client_id=310):
    for enc_id in range(start_encounter_id, start_encounter_id + num_new):
        client_ref = random.randint(1, max_client_id)
        clinician_ref = random.randint(1, 7)
        unit_ref = clinician_to_unit[clinician_ref]

        enc_date = random_date()
        enc_time = random_time()

        triage_level = random.randint(1, 5)
        pain_score = random.randint(0, 10)

        temperature_c = round(random.uniform(35.5, 40.5), 1)
        bp_sys = random.randint(90, 190)
        bp_dia = random.randint(50, 110)

        cursor.execute("""
            INSERT INTO clinical_encounters (
                encounter_id, client_ref_id, clinician_ref_id, unit_ref_id,
                encounter_date, encounter_time,
                triage_level, pain_score,
                temperature_c, bp_systolic, bp_diastolic
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """, (
            enc_id, client_ref, clinician_ref, unit_ref,
            enc_date, enc_time,
            triage_level, pain_score,
            temperature_c, bp_sys, bp_dia
        ))

    print(f"Inserted {num_new} encounter records.")

cursor.execute("DELETE FROM clinical_encounters;")
connection.commit()
generate_encounters(cursor, start_encounter_id=1, num_new=1015, max_client_id=310)
connection.commit()

# 7. Generate diagnostic events 

def generate_diagnoses(cursor, num_encounters=1015):
    diag_count = 0
    cursor.execute("DELETE FROM diagnostic_events;")
    connection.commit()

    for enc_id in range(1, num_encounters + 1):
        num_diags = random.choice([0, 1, 1, 2, 3])
        for _ in range(num_diags):
            code, label = random.choice(diagnosis_codes)
            severity = random.randint(1, 3)
            note = f"{label} (severity band {severity})"

            cursor.execute("""
                INSERT INTO diagnostic_events (
                    encounter_ref_id, diagnostic_code, severity_band, clinical_notes
                )
                VALUES (?, ?, ?, ?);
            """, (enc_id, code, severity, note))
            diag_count += 1

    print(f"Inserted {diag_count} diagnostic events.")

generate_diagnoses(cursor, num_encounters=1015)
connection.commit()

# 8. Generate procedures 

def generate_procedures(cursor, num_encounters=1015):
    proc_count = 0
    cursor.execute("DELETE FROM procedure_catalogue;")
    connection.commit()

    for enc_id in range(1, num_encounters + 1):
        num_procs = random.choice([0, 0, 1, 1, 2])
        if num_procs == 0:
            continue

        chosen = random.sample(procedure_list, k=num_procs)
        for code, label, base_cost in chosen:
            factor = random.uniform(0.8, 1.2)
            cost = round(base_cost * factor, 2)

            cursor.execute("""
                INSERT OR IGNORE INTO procedure_catalogue (
                    encounter_ref_id, procedure_code, procedure_label, cost_in_currency
                )
                VALUES (?, ?, ?, ?);
            """, (enc_id, code, label, cost))

            proc_count += 1

    print(f"Inserted ~{proc_count} procedure records.")

generate_procedures(cursor, num_encounters=1015)
connection.commit()

# 9. Generate medications 

def generate_medications(cursor, num_encounters=1015):
    med_count = 0
    cursor.execute("DELETE FROM medication_orders;")
    connection.commit()

    for enc_id in range(1, num_encounters + 1):
        num_meds = random.choice([0, 0, 1, 1, 2, 2, 3])
        if num_meds == 0:
            continue

        chosen = random.sample(drug_list, k=num_meds)
        for drug in chosen:
            if "Injection" in drug or "injection" in drug.lower():
                dose = random.uniform(2, 20)
            elif "inhaler" in drug.lower():
                dose = random.uniform(50, 200)
            else:
                dose = random.uniform(100, 1000)

            duration = random.randint(3, 14)

            cursor.execute("""
                INSERT OR IGNORE INTO medication_orders (
                    encounter_ref_id, drug_label, dose_in_mg, duration_days
                )
                VALUES (?, ?, ?, ?);
            """, (enc_id, drug, round(dose, 1), duration))
            med_count += 1

    print(f"Inserted ~{med_count} medication records.")

generate_medications(cursor, num_encounters=1015)
connection.commit()

# 10. Generate billing 

def generate_billing(cursor, num_encounters=1015):
    bill_count = 0
    cursor.execute("DELETE FROM financial_statements;")
    connection.commit()

    for enc_id in range(1, num_encounters + 1):
        cursor.execute("""
            SELECT COALESCE(SUM(cost_in_currency), 0)
            FROM procedure_catalogue
            WHERE encounter_ref_id = ?;
        """, (enc_id,))
        proc_total = cursor.fetchone()[0]

        cursor.execute("""
            SELECT COUNT(*)
            FROM medication_orders
            WHERE encounter_ref_id = ?;
        """, (enc_id,))
        med_count = cursor.fetchone()[0]
        med_total = med_count * 15.0

        base_fee = random.uniform(40, 80)
        amount = round(base_fee + proc_total + med_total, 2)

        if random.random() < 0.85:
            payment_flag = "Paid"
            cursor.execute("""
                SELECT encounter_date FROM clinical_encounters WHERE encounter_id = ?;
            """, (enc_id,))
            row = cursor.fetchone()
            if row:
                enc_date = dt.date.fromisoformat(row[0])
                offset_days = random.randint(0, 7)
                settle_date = (enc_date + dt.timedelta(days=offset_days)).isoformat()
            else:
                settle_date = None
        else:
            payment_flag = "Pending"
            settle_date = None

        cursor.execute("""
            INSERT INTO financial_statements (
                encounter_ref_id, amount_payable, payment_flag, settlement_date
            )
            VALUES (?, ?, ?, ?);
        """, (enc_id, amount, payment_flag, settle_date))

        bill_count += 1

    print(f"Inserted {bill_count} billing records.")

generate_billing(cursor, num_encounters=1015)
connection.commit()

# 11. Show final row counts

tables = [
    "care_units",
    "clinician_registry",
    "client_profiles",
    "clinical_encounters",
    "diagnostic_events",
    "procedure_catalogue",
    "medication_orders",
    "financial_statements"
]

print("\nFinal table row counts:")
for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t};")
    count = cursor.fetchone()[0]
    print(f"{t:22s} {count}")

# 12. Close connection 

connection.close()
print("\nConnection closed.")


Connected to: C:\M\sqlite project\Hospital_Data_base.db
Schema created.
Reference data inserted.
Inserted 310 client records.
Inserted 1015 encounter records.
Inserted 1443 diagnostic events.
Inserted ~804 procedure records.
Inserted ~1313 medication records.
Inserted 1015 billing records.

Final table row counts:
care_units             5
clinician_registry     7
client_profiles        310
clinical_encounters    1015
diagnostic_events      1443
procedure_catalogue    804
medication_orders      1313
financial_statements   1015

Connection closed.
